# 🩺 Predictive Pulse — Exploratory Data Analysis (EDA)

This notebook explores the hypertension dataset to understand feature distributions,
relationships, and class balance before model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 6)

# Load raw data
df = pd.read_csv('../data/raw/hypertension_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

## 1. Dataset Overview

In [ ]:
print('=== Data Types ===')
print(df.dtypes)
print(f'\n=== Missing Values ===')
print(df.isnull().sum())
print(f'\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
df.info()

## 2. Univariate Analysis

In [ ]:
# Numerical feature distributions
num_cols = ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol', 'glucose']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3, i % 3]
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.1f}')
    ax.legend()
plt.tight_layout()
plt.savefig('../docs/eda_numerical_distributions.png', dpi=150)
plt.show()

In [ ]:
# Categorical feature distributions
cat_cols = ['gender', 'smoking', 'alcohol', 'physical_activity']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for i, col in enumerate(cat_cols):
    sns.countplot(data=df, x=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col} Distribution')
    axes[i].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../docs/eda_categorical_distributions.png', dpi=150)
plt.show()

## 3. Target Variable Analysis

In [ ]:
# Class distribution
stage_order = ['Normal', 'Elevated', 'Stage 1', 'Stage 2', 'Crisis']
stage_colors = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c', '#8b0000']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
order_present = [s for s in stage_order if s in df['hypertension_stage'].values]
colors_present = [stage_colors[stage_order.index(s)] for s in order_present]
sns.countplot(data=df, x='hypertension_stage', order=order_present, 
              palette=colors_present, ax=axes[0])
axes[0].set_title('Hypertension Stage Distribution')
axes[0].set_xlabel('Stage')
axes[0].set_ylabel('Count')

# Pie chart
stage_counts = df['hypertension_stage'].value_counts()
stage_counts = stage_counts.reindex([s for s in stage_order if s in stage_counts.index])
axes[1].pie(stage_counts.values, labels=stage_counts.index, autopct='%1.1f%%',
            colors=[stage_colors[stage_order.index(s)] for s in stage_counts.index])
axes[1].set_title('Class Distribution (%)')

plt.tight_layout()
plt.savefig('../docs/eda_target_distribution.png', dpi=150)
plt.show()

print('Class counts:')
print(df['hypertension_stage'].value_counts())

In [ ]:
# Mean feature values per hypertension stage
numeric_df = df[num_cols + ['hypertension_stage']].copy()
stage_means = numeric_df.groupby('hypertension_stage').mean()
stage_means = stage_means.reindex([s for s in stage_order if s in stage_means.index])
print('Mean feature values per stage:')
stage_means.round(1)

## 4. Bivariate & Multivariate Analysis

In [ ]:
# Correlation heatmap
numeric_only = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_only.corr(), annot=True, cmap='RdBu_r', center=0, fmt='.2f',
            square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.savefig('../docs/eda_correlation_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Box plots per hypertension stage
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, col in enumerate(num_cols):
    ax = axes[i // 3, i % 3]
    order_present = [s for s in stage_order if s in df['hypertension_stage'].values]
    sns.boxplot(data=df, x='hypertension_stage', y=col, order=order_present, 
                palette=colors_present, ax=ax)
    ax.set_title(f'{col} by Stage')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../docs/eda_boxplots_by_stage.png', dpi=150)
plt.show()

In [ ]:
# BP scatter plot colored by stage
plt.figure(figsize=(10, 8))
for stage, color in zip(stage_order, stage_colors):
    mask = df['hypertension_stage'] == stage
    if mask.sum() > 0:
        plt.scatter(df.loc[mask, 'systolic_bp'], df.loc[mask, 'diastolic_bp'],
                   c=color, label=stage, alpha=0.5, s=20)

plt.xlabel('Systolic BP (mmHg)')
plt.ylabel('Diastolic BP (mmHg)')
plt.title('Blood Pressure Distribution by Stage')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/eda_bp_scatter.png', dpi=150)
plt.show()

## 5. Key Findings

1. **Blood Pressure** is the strongest predictor of hypertension stage (by design, as stages are defined by BP thresholds)
2. **Age** and **BMI** show positive correlation with higher BP levels
3. **Class imbalance** is present — Stage 2 and Stage 1 are more common than Crisis and Elevated
4. **Lifestyle factors** (smoking, alcohol, physical activity) show expected associations with higher BP
5. **Diabetes** is associated with higher systolic and diastolic BP readings
6. **Gender** shows males have slightly higher BP on average

### Implications for Modeling
- SMOTE or class weighting needed to handle class imbalance
- Feature engineering (pulse pressure, BMI/age categories) may improve classification
- Tree-based models likely to perform well given the threshold-based nature of the target